# God-Tier RBX1 30-Seed Binder Design Campaign

**Target:** RBX1 (PDB 2LGV), sequence-only, 65-residue binders
**Pipeline:** Protenix2025 design → Boltz2 validation → God-tier filtering
**Strategy:** 3 notebooks × 10 seeds each (change `NOTEBOOK_ID` below)

### Before running:
1. Upload `design_god.py`, `validate_boltz2.py`, `filter_god.py` to `My Drive/mosaic_god_campaign/scripts/`
2. Set `NOTEBOOK_ID` to 1, 2, or 3
3. Run all cells top to bottom
4. After ALL 3 notebooks finish, run the filter cell from any one notebook

In [ ]:
# ============================================================
# CONFIG — Change NOTEBOOK_ID (1, 2, or 3)
# ============================================================

NOTEBOOK_ID = 1          # <-- CHANGE THIS: 1, 2, or 3

SEEDS_PER_NOTEBOOK = 10
SEED_START = (NOTEBOOK_ID - 1) * SEEDS_PER_NOTEBOOK
SEED_END = SEED_START + SEEDS_PER_NOTEBOOK
SEEDS = list(range(SEED_START, SEED_END))

# Google Drive paths
DRIVE_BASE = "/content/drive/MyDrive/mosaic_god_campaign"
SCRIPTS_DIR = f"{DRIVE_BASE}/scripts"
OUTPUT_DIR = f"{DRIVE_BASE}/outputs"

# RBX1 target sequence (108aa construct, NOT the shorter PDB crystal sequence)
TARGET_SEQUENCE = "MAAAMDVDTPSGTNSGAGKKRFEVKKWNAVALWAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHCISRWLKTRQVCPLDNREWEFQKYGH"

print(f"Notebook {NOTEBOOK_ID}: seeds {SEED_START}–{SEED_END - 1}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# SETUP — Install dependencies
# ============================================================

import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# JAX with CUDA (Colab may already have this, but ensure correct version)
pip_install("jax[cuda12]")

# mosaic — the umbrella package (Protenix2025, ProteinMPNN, Boltz2, losses)
# If this is a private repo, prefix with token:
#   pip_install("mosaic[all] @ git+https://<TOKEN>@github.com/escalante-bio/mosaic.git")
pip_install("mosaic[all] @ git+https://github.com/escalante-bio/mosaic.git")

# gemmi for PDB handling (needed by validate_boltz2.py)
pip_install("gemmi")

print("\nDependencies installed.")

In [ ]:
# ============================================================
# GPU CHECK
# ============================================================

import jax

devices = jax.devices()
print(f"JAX devices: {devices}")

gpu = [d for d in devices if d.device_kind == "gpu"]
if not gpu:
    raise RuntimeError(
        "No GPU detected! Go to Runtime > Change runtime type > A100 GPU"
    )

print(f"GPU ready: {gpu[0]}")

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE + CREATE DIRECTORIES
# ============================================================

from google.colab import drive
import os

drive.mount("/content/drive")

os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify scripts are uploaded
required_scripts = ["design_god.py", "validate_boltz2.py", "filter_god.py"]
missing = [s for s in required_scripts if not os.path.exists(f"{SCRIPTS_DIR}/{s}")]

if missing:
    print(f"ERROR: Missing scripts in {SCRIPTS_DIR}:")
    for s in missing:
        print(f"  - {s}")
    print(f"\nUpload them to Google Drive at: My Drive/mosaic_god_campaign/scripts/")
    raise FileNotFoundError(f"Missing: {missing}")

print(f"All scripts found in {SCRIPTS_DIR}")
print(f"Outputs will be saved to {OUTPUT_DIR}")

---
## Phase 1: Design (Protenix2025 + ProteinMPNN)

Runs `design_god.py` for each seed. Each seed takes ~15-25 min (first seed longer due to JIT).

**Checkpointed:** Re-running skips seeds that already have `csv_line.txt`.

In [ ]:
# ============================================================
# PHASE 1: DESIGN LOOP
# ============================================================

import subprocess
import sys
import os
import time

design_script = f"{SCRIPTS_DIR}/design_god.py"
design_results = {}  # seed -> {sequence, length, cys_count, binder_plddt, target_plddt, iptm}
design_failures = []

total = len(SEEDS)
campaign_start = time.time()

for i, seed in enumerate(SEEDS):
    seed_dir = f"{OUTPUT_DIR}/seed_{seed:03d}"
    csv_path = f"{seed_dir}/csv_line.txt"

    # Checkpoint: skip if already done
    if os.path.exists(csv_path):
        line = open(csv_path).read().strip()
        parts = line.split(",")
        design_results[seed] = {
            "sequence": parts[1], "length": int(parts[2]),
            "cys_count": int(parts[3]), "binder_plddt": float(parts[4]),
            "target_plddt": float(parts[5]), "iptm": float(parts[6]),
        }
        print(f"[{i+1}/{total}] Seed {seed}: CACHED (pLDDT={parts[4]}, ipTM={parts[6]})")
        continue

    os.makedirs(seed_dir, exist_ok=True)
    print(f"\n[{i+1}/{total}] Seed {seed}: designing...", flush=True)
    t0 = time.time()

    try:
        result = subprocess.run(
            [sys.executable, design_script, str(seed)],
            cwd=seed_dir,
            capture_output=True,
            text=True,
            timeout=3600,  # 1hr max per seed
        )

        # Print last 10 lines of output for progress
        out_lines = result.stdout.strip().split("\n")
        for line in out_lines[-10:]:
            print(f"  {line}")

        if result.returncode != 0:
            print(f"  ERROR (exit {result.returncode}):")
            err_lines = result.stderr.strip().split("\n")
            for line in err_lines[-5:]:
                print(f"  {line}")
            design_failures.append(seed)
            continue

        # Extract CSV line from stdout
        csv_line = None
        for line in out_lines:
            if line.startswith("CSV:"):
                csv_line = line[4:]  # strip "CSV:" prefix
                break

        if csv_line is None:
            print(f"  ERROR: No CSV line in output")
            design_failures.append(seed)
            continue

        # Save CSV line for filter_god.py
        with open(csv_path, "w") as f:
            f.write(csv_line)

        parts = csv_line.split(",")
        design_results[seed] = {
            "sequence": parts[1], "length": int(parts[2]),
            "cys_count": int(parts[3]), "binder_plddt": float(parts[4]),
            "target_plddt": float(parts[5]), "iptm": float(parts[6]),
        }

        elapsed = time.time() - t0
        print(f"  Done in {elapsed/60:.1f}min — pLDDT={parts[4]}, ipTM={parts[6]}")

    except subprocess.TimeoutExpired:
        print(f"  TIMEOUT (>1hr)")
        design_failures.append(seed)
    except Exception as e:
        print(f"  EXCEPTION: {e}")
        design_failures.append(seed)

# Summary
elapsed_total = (time.time() - campaign_start) / 60
print(f"\n{'='*60}")
print(f"Phase 1 complete: {len(design_results)}/{total} seeds designed ({elapsed_total:.0f}min)")
if design_failures:
    print(f"Failed seeds: {design_failures}")
print(f"{'='*60}")

---
## Phase 2: Boltz2 Validation

Runs `validate_boltz2.py` for each successful design. Uses best-of-5 sampling with 8 recycling steps.

**Checkpointed:** Re-running skips seeds that already have validation results.

In [ ]:
# ============================================================
# CLEAR GPU MEMORY BEFORE VALIDATION
# ============================================================

import gc

# Force garbage collection to free Protenix2025 model from GPU
gc.collect()

# JAX doesn't have explicit GPU memory release, but the Boltz2 model
# will allocate into freed space. If OOM occurs, restart runtime
# and re-run from this cell (Phase 1 results are checkpointed on Drive).

print("GPU memory cleared (gc.collect). Starting Boltz2 validation...")

In [ ]:
# ============================================================
# PHASE 2: BOLTZ2 VALIDATION LOOP
# ============================================================

import subprocess
import sys
import os
import csv
import time

validate_script = f"{SCRIPTS_DIR}/validate_boltz2.py"
boltz2_csv_path = f"{OUTPUT_DIR}/boltz2_god_summary.csv"

# Load existing Boltz2 results (checkpoint)
existing_b2 = set()
if os.path.exists(boltz2_csv_path):
    with open(boltz2_csv_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_b2.add(int(row["seed"]))
    print(f"Found {len(existing_b2)} existing Boltz2 results")

# Reload design results from Drive (in case runtime was restarted)
if not design_results:
    print("Reloading design results from Drive...")
    design_results = {}
    for seed in SEEDS:
        csv_path = f"{OUTPUT_DIR}/seed_{seed:03d}/csv_line.txt"
        if os.path.exists(csv_path):
            parts = open(csv_path).read().strip().split(",")
            design_results[seed] = {
                "sequence": parts[1], "length": int(parts[2]),
                "cys_count": int(parts[3]), "binder_plddt": float(parts[4]),
                "target_plddt": float(parts[5]), "iptm": float(parts[6]),
            }
    print(f"Loaded {len(design_results)} designs")

# Seeds to validate
seeds_to_validate = [s for s in sorted(design_results.keys()) if s not in existing_b2]
total = len(seeds_to_validate)
print(f"Validating {total} seeds with Boltz2...\n")

validation_results = []
validation_failures = []
val_start = time.time()

for i, seed in enumerate(seeds_to_validate):
    d = design_results[seed]
    seed_dir = f"{OUTPUT_DIR}/seed_{seed:03d}"

    print(f"[{i+1}/{total}] Seed {seed}: validating with Boltz2...", flush=True)
    t0 = time.time()

    try:
        result = subprocess.run(
            [sys.executable, validate_script, str(seed), d["sequence"], TARGET_SEQUENCE],
            cwd=seed_dir,
            capture_output=True,
            text=True,
            timeout=3600,
        )

        out_lines = result.stdout.strip().split("\n")
        for line in out_lines[-8:]:
            print(f"  {line}")

        if result.returncode != 0:
            print(f"  ERROR (exit {result.returncode}):")
            err_lines = result.stderr.strip().split("\n")
            for line in err_lines[-5:]:
                print(f"  {line}")
            validation_failures.append(seed)
            continue

        # Extract CSV line
        csv_line = None
        for line in out_lines:
            if line.startswith("CSV:"):
                csv_line = line[4:]
                break

        if csv_line is None:
            print(f"  ERROR: No CSV line in output")
            validation_failures.append(seed)
            continue

        parts = csv_line.split(",")
        validation_results.append({
            "seed": int(parts[0]),
            "boltz2_plddt": parts[1],
            "boltz2_iptm": parts[2],
            "boltz2_bt_ipsae": parts[3],
            "boltz2_tb_ipsae": parts[4],
            "boltz2_min_ipsae": parts[5],
        })

        elapsed = time.time() - t0
        print(f"  Done in {elapsed/60:.1f}min — ipTM={parts[2]}, min_ipSAE={parts[5]}")

    except subprocess.TimeoutExpired:
        print(f"  TIMEOUT (>1hr)")
        validation_failures.append(seed)
    except Exception as e:
        print(f"  EXCEPTION: {e}")
        validation_failures.append(seed)

# Append new results to boltz2 summary CSV
if validation_results:
    write_header = not os.path.exists(boltz2_csv_path)
    with open(boltz2_csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "seed", "boltz2_plddt", "boltz2_iptm",
            "boltz2_bt_ipsae", "boltz2_tb_ipsae", "boltz2_min_ipsae",
        ])
        if write_header:
            writer.writeheader()
        writer.writerows(validation_results)
    print(f"\nAppended {len(validation_results)} results to {boltz2_csv_path}")

elapsed_total = (time.time() - val_start) / 60
print(f"\n{'='*60}")
print(f"Phase 2 complete: {len(validation_results)}/{total} validated ({elapsed_total:.0f}min)")
if validation_failures:
    print(f"Failed seeds: {validation_failures}")
print(f"{'='*60}")

---
## Phase 3: Filter + Rank (Run ONCE after all 3 notebooks finish)

Only run this cell after notebooks 1, 2, and 3 have all completed Phases 1 and 2.
It reads all 30 seed results from Drive and applies god-tier filtering.

In [ ]:
# ============================================================
# PHASE 3: FILTER + RANK (run once after all 3 notebooks done)
# ============================================================

import subprocess
import sys
import os

filter_script = f"{SCRIPTS_DIR}/filter_god.py"
output_csv = f"{OUTPUT_DIR}/god_filtered_candidates.csv"

# Count available results
design_count = len([d for d in os.listdir(OUTPUT_DIR)
                    if d.startswith("seed_") and
                    os.path.exists(f"{OUTPUT_DIR}/{d}/csv_line.txt")])
print(f"Found {design_count} design results in {OUTPUT_DIR}")

if os.path.exists(boltz2_csv_path):
    with open(boltz2_csv_path) as f:
        b2_count = sum(1 for _ in f) - 1  # subtract header
    print(f"Found {b2_count} Boltz2 validations")
else:
    print(f"WARNING: No Boltz2 summary CSV found at {boltz2_csv_path}")

if design_count < 30:
    print(f"\nWARNING: Only {design_count}/30 designs found. Run all 3 notebooks first!")
    print("Proceeding with available results...\n")

# Run filter
result = subprocess.run(
    [
        sys.executable, filter_script,
        "--design-dir", OUTPUT_DIR,
        "--boltz2-csv", boltz2_csv_path,
        "--output", output_csv,
    ],
    capture_output=True,
    text=True,
)

# Display full filter output
print(result.stdout)

if result.returncode != 0:
    print(f"Filter failed (exit {result.returncode}):")
    print(result.stderr)
else:
    if os.path.exists(output_csv):
        print(f"\nFiltered candidates saved to: {output_csv}")
        print(f"(Google Drive path: My Drive/mosaic_god_campaign/outputs/god_filtered_candidates.csv)")

---
## Quick Status Check

Run this anytime to see what's been completed across all notebooks.

In [ ]:
# ============================================================
# STATUS CHECK — What's done across all 3 notebooks?
# ============================================================

import os

print(f"Output directory: {OUTPUT_DIR}\n")

for nb_id in [1, 2, 3]:
    start = (nb_id - 1) * 10
    end = start + 10
    designed = 0
    validated = 0
    for s in range(start, end):
        seed_dir = f"{OUTPUT_DIR}/seed_{s:03d}"
        if os.path.exists(f"{seed_dir}/csv_line.txt"):
            designed += 1
        if os.path.exists(f"{seed_dir}/rbx1_boltz2_seed{s:03d}.pdb"):
            validated += 1
    marker = " <-- this notebook" if nb_id == NOTEBOOK_ID else ""
    print(f"  Notebook {nb_id} (seeds {start:>2d}–{end-1:>2d}): "
          f"{designed:>2d}/10 designed, {validated:>2d}/10 validated{marker}")

# Overall
total_designed = len([d for d in os.listdir(OUTPUT_DIR)
                      if d.startswith("seed_") and
                      os.path.exists(f"{OUTPUT_DIR}/{d}/csv_line.txt")])
print(f"\n  Total: {total_designed}/30 designed")

if os.path.exists(f"{OUTPUT_DIR}/god_filtered_candidates.csv"):
    with open(f"{OUTPUT_DIR}/god_filtered_candidates.csv") as f:
        n_filtered = sum(1 for _ in f) - 1
    print(f"  Filtered candidates: {n_filtered}")
else:
    print(f"  Filtering: not yet run")